# TP 1 — La naturaleza de la econometría y los datos económicos (Python)

**Alumna:** Lucía Vázquez · **Materia:** Econometría I · **Fecha:** 31/08/2026
**Referencia:** Wooldridge, *Introducción a la econometría*, capítulo 1

Este notebook resuelve en Python los ejercicios 1.2, 1.5, 1.6, 1.7 y 1.8.
El desarrollo teórico completo (pregunta económica, especificación,
inferencia, interpretación y limitaciones, según la plantilla de entrega)
está en `TP1_entrega.docx`; aquí se documentan el código, los resultados
numéricos y una síntesis breve de cada respuesta. La versión en R
equivalente está en `TP1_R.R`.

In [1]:
import numpy as np
import pandas as pd
import wooldridge as woo
import statsmodels.formula.api as smf
from scipy import stats

pd.set_option('display.precision', 4)


## Ejercicio 1.2 (Wooldridge C1.2, `bwght`)
Cuántas mujeres hay en la muestra, cuántas fumaron durante el embarazo, el
promedio de `cigs` y ese promedio solo entre fumadoras.

In [2]:
bwght = woo.dataWoo('bwght')

n = len(bwght)
fumadoras = (bwght['cigs'] > 0).sum()
media_todas = bwght['cigs'].mean()
media_fumadoras = bwght.loc[bwght['cigs'] > 0, 'cigs'].mean()

print('Tamaño de la muestra (mujeres):', n)
print('Mujeres que fumaron durante el embarazo:', fumadoras)
print('Promedio de cigs, toda la muestra:', round(media_todas, 3))
print('Promedio de cigs, solo fumadoras:', round(media_fumadoras, 3))


Tamaño de la muestra (mujeres): 1388
Mujeres que fumaron durante el embarazo: 212
Promedio de cigs, toda la muestra: 2.087
Promedio de cigs, solo fumadoras: 13.665


**Síntesis:** 1,388 mujeres, 212 (15.3%) fumaron durante el embarazo. El
promedio general (2.09) diluye el 84.7% de ceros; el promedio condicional
(13.67) sí describe la intensidad de consumo de quien fuma, y es el que
responde la pregunta planteada. Ambos números probablemente subestiman el
consumo real por subdeclaración de una conducta estigmatizada.

## Ejercicio 1.5 (`wage1`)
Salario promedio por hora de varones y mujeres, y tres variables del
dataset que podrían explicar parte de la brecha sin invocar
discriminación.

In [3]:
wage1 = woo.dataWoo('wage1')

medias = wage1.groupby('female')['wage'].mean()
brecha_abs = medias[0] - medias[1]
brecha_pct = 100 * brecha_abs / medias[0]
print(medias)
print(f'\nBrecha: {brecha_abs:.3f} USD/hora ({brecha_pct:.1f}% del salario de los varones)')

print()
print(wage1.groupby('female')[['educ', 'exper', 'tenure']].mean())
print()
print(wage1.groupby('female')[['profocc', 'clerocc', 'servocc']].mean())

m = smf.ols('wage ~ female', data=wage1).fit()
print('\nwage ~ female:', dict(m.params.round(4)), 'SE:', dict(m.bse.round(4)),
      'n =', int(m.nobs), 'R2 =', round(m.rsquared, 3))
t, p = stats.ttest_ind(wage1.loc[wage1.female==1,'wage'], wage1.loc[wage1.female==0,'wage'])
print('t-test H0: misma media =>  t =', round(t,3), ' p =', p)


female
0    7.0995
1    4.5877
Name: wage, dtype: float64

Brecha: 2.512 USD/hora (35.4% del salario de los varones)

           educ    exper  tenure
female                          
0       12.7883  17.5584  6.4745
1       12.3175  16.4286  3.6151

        profocc  clerocc  servocc
female                           
0        0.4489   0.0401   0.0876
1        0.2778   0.3056   0.1984

wage ~ female: {'Intercept': np.float64(7.0995), 'female': np.float64(-2.5118)} SE: {'Intercept': np.float64(0.21), 'female': np.float64(0.3034)} n = 526 R2 = 0.116
t-test H0: misma media =>  t = -8.279  p = 1.0417636408499264e-15


**Síntesis:** brecha de 2.51 USD/hora (35.4%), estadísticamente distinguible
de cero (p ≈ 1e-15). Tres variables que explican parte de ella sin invocar
discriminación: **`tenure`** (las mujeres promedian menos antigüedad: 3.62 vs.
6.47 años), la **distribución ocupacional** (más concentración femenina en
clerical/servicios y menos en profesional) y **`exper`** (16.43 vs. 17.56
años). Controlando por `tenure`, el coeficiente de `female` baja de -2.51 a
-2.09: parte de la brecha bruta refleja la menor antigüedad, no
necesariamente un trato distinto en el mismo puesto.

## Ejercicio 1.6 (aproximación logarítmica)
Cambio porcentual exacto vs. aproximación logarítmica: 6,000 vs. 5,500
dólares de gasto por alumno, y 5,500 vs. 11,000.

In [4]:
def cambio(x0, x1):
    exacto = 100 * (x1 - x0) / x0
    aproximado = 100 * (np.log(x1) - np.log(x0))
    return exacto, aproximado, aproximado - exacto

for x0, x1, etiqueta in [(5500, 6000, '5,500 a 6,000'),
                          (5500, 11000, '5,500 a 11,000')]:
    exacto, aproximado, error = cambio(x0, x1)
    print(f'{etiqueta}: exacto={exacto:.2f}%, log={aproximado:.2f}%, error={error:.2f} p.p.')


5,500 a 6,000: exacto=9.09%, log=8.70%, error=-0.39 p.p.
5,500 a 11,000: exacto=100.00%, log=69.31%, error=-30.69 p.p.


**Síntesis:** para el cambio chico (9.09% real) el error es de apenas 0.39
puntos porcentuales; para el cambio grande (100% real) el error es de casi
31 puntos porcentuales, y la aproximación siempre **subestima** los
aumentos y **sobrestima la magnitud** de las caídas (porque `log` es
cóncavo). Regla práctica: usar la aproximación hasta cambios de 10-15%;
para cambios mayores, reportar el cambio exacto.

## Ejercicio 1.7 (estructura de datos)
Clasificación de cuatro conjuntos de datos hipotéticos. No requiere código:
la unidad de observación y la garantía (o no) de seguimiento de las mismas
unidades en el tiempo alcanzan para clasificar.

**a) Dólar mayorista diario, 2015–2025:** serie de tiempo — una sola unidad
seguida repetidamente en el tiempo.

**b) EPH 2T-2024, 3,000 hogares:** corte transversal — muchos hogares, un
solo momento.

**c) PBI/población/esperanza de vida, 180 países, 1990-2000-2010-2020:**
panel — mismas 180 unidades seguidas en varios momentos.

**d) Ventas y empleo, 500 empresas, 2019 y 2023, sin garantía de que sean
las mismas:** corte transversal combinado — dos momentos, pero muestras
independientes, no seguimiento de unidades.

## Ejercicio 1.8 (Wooldridge, problema 1.2 — capacitación y productividad)
Problema conceptual, sin datos ni código.

**a) Experimento subyacente:** asignar aleatoriamente la capacitación entre
trabajadores/empresas idénticos en todo lo demás (capital humano,
tecnología, gestión, mercado) y comparar productividad ex post.

**b) ¿Es independiente la decisión de capacitar?** No, en general. Rompen la
independencia: educación previa y antigüedad (medibles); motivación/habilidad
innata y calidad de gestión (no medibles).

**c) Factor ajeno a los trabajadores:** calidad del capital físico/tecnología
de la empresa.

**d) ¿La correlación positiva establece causalidad?** No: es compatible con
causalidad inversa (empresas más productivas capacitan más porque tienen
recursos ociosos) y con una variable omitida (calidad gerencial) que afecta
ambas cosas a la vez.